# Demo Notebook for lca_supplymat

This notebook demonstrates the usage of `lca_supplymat` :

This library setup new LCIA impact methods for Brightway to compute the amount of manufactured materials (copper, concrete, steel, ...) for an inventory.

## Installation

This package is available on `Pypi` :
```bash
pip install lca_supplymat
```

## Imports

In [1]:
import bw2data
from bw2io import import_ecoinvent_release
from bw2data import projects
from bw2data import databases
from bw2data.database import Database
import bw2calc

from lca_supplymat import setup_lca_supply_mat, cleanup, MATERIALS_METHOD_NAME, list_methods_available_methods

## Prerequisite - Setup a brightway project with ecoinvent

In [2]:
projects.set_current("lca_supplymat_test")

In [ ]:
import_ecoinvent_release(
    version="3.11",
    system_model="cutoff",
    username="YOUR_ECOINVENT_USERNAME",
    password="YOUR_PASSWORD")

In [3]:
# List of databases
list(print(name) for name in databases);

ecoinvent-3.11-biosphere
ecoinvent-3.11-cutoff


## Install of material LCIA methods

The following code will setup materials methods on your background database :
- It will create a separate database with elementary fluxes called `lca_supplymat`
- It will compute the net production of the requested materials for each activity and setup additional exchanges to account for it
- It will create a new Brightway Method per material

The library comes with many definitions for materials, [listed here](https://git.sophia.minesparis.psl.eu/oie/acv/lca-supplymat/-/blob/main/lca_supplymat/res/materials_methods.yaml?ref_type=heads). You may also import your own by providing the path to the parameter `yaml_file`.



In [4]:
# List available methods in the bundled file
list_methods_available_methods()

['concrete',
 'cement',
 'gravel',
 'steel-iron',
 'aluminium',
 'copper',
 'rare-earths-oxide',
 'titanium',
 'zirconium-oxide',
 'platinum',
 'iridium',
 'nickel-class1',
 'silicon-metal',
 'silver',
 'lithium-carbonate',
 'lithium-hydroxide',
 'cobalt-sulfate',
 'nickel-sulfate',
 'manganese-sulfate',
 'graphite',
 'graphite-nat',
 'zirconium-sponge',
 'iron-scrap']

In [5]:
# We strongly advice to clean-up the methods before the set-up. 
# Making the set-up several times for the same materials without cleaning might indeed multiply the impact values.
cleanup('ecoinvent-3.11-cutoff')

In [6]:
# Setup impact methods for a couple of definitions
# REMINDER : we advice to cleanup before installing
setup_lca_supply_mat(
    target_db='ecoinvent-3.11-cutoff',
    out_folder="data/out", # Write CSV report for each material method
    subset=["concrete", "copper"]) # the definitions you want to import (all by default)


# WARNING : if you do the set up twice using this notebook (cleanup 1, set-up 1, cleanup 2, set up 2), 
#you might have an error when doing the set up n°2. In this case, just restart the kernel before setup 2.  

setting dummy counting exchanges ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:10

INFO     Created method Brightway2 Method: lca_supplymat: total: copper key: ('lca_supplymat', 'total', 'copper') unit :
         [kilogram]                                                                                                     
INFO     Checking impact values for all activities, for materials : concrete                                            
INFO     Saving statistics about material method in data/out\concrete.csv                                               
INFO     Checking impact values for all activities, for materials : copper                                              
INFO     Saving statistics about material method in data/out\copper.csv                                                 


## Use the new method in the script or in Activity Browser as any LCIA method implemented in Brightway.

## Example 
### List installed methods

In [7]:
# List available material methods
methods = list(m for m in bw2data.methods if m[0] == MATERIALS_METHOD_NAME)
methods

[('lca_supplymat', 'total', 'concrete'), ('lca_supplymat', 'total', 'copper')]

### Find a wind turbine

In [8]:
# Find activity producing a wind turbine
turbine_act = Database('ecoinvent-3.11-cutoff').search("wind turbine construction, 2.3MW, precast concrete tower, onshore")[0]
turbine_act

'wind turbine construction, 2.3MW, precast concrete tower, onshore' (unit, CA-QC, None)

### Compute the amount of concrete and  copper in the wind turbine

In [9]:
# Compute regular LCIA with Brightway
fu = {"turbine" : {turbine_act.id : 1.0}}
meth_cfg = {"impact_categories": methods}
data_objs = bw2data.get_multilca_data_objs(
    functional_units=fu,
    method_config=meth_cfg)
lca = bw2calc.MultiLCA(demands=fu, method_config=meth_cfg, data_objs=data_objs)
lca.lci()
lca.lcia()

for (key, value), unit in zip(lca.scores.items(), ["cubic_meter", "kg"]) :
    print(f"{key[0][2]} : {value:.1f} [{unit}]")

concrete : 662.7 [cubic_meter]
copper : 11429.9 [kg]


## Advanced : Writing your own definitions

You can provide your own definitions by passing the `yaml_file` parameter to the setup function.

The file maps a definition key to a set of matching rules:

```yaml
copper:
  matchings: ["copper, cathode*", "copper, anode*"]
  excludes: ["*scrap*"]      # optional: patterns that veto a match
  unit: kilogram
  tolerance: 0.05            # optional: allowed deviation for the impact check (default 0.05)
```

- **`matchings`** (required): a list of patterns used to select activities.
- **`excludes`** (optional): a list of patterns; if any matches, the activity is rejected.
- **`unit`** (required): the unit the matching activities must be in.
- **`tolerance`** (optional): allowed deviation of the impact score from 1 during the `check_impacts` step (default `0.05`).

**Matching semantics.** Matching is performed against an activity's **reference product name** (`act["reference product"]`), and the activity's unit must equal the definition's `unit`. A pattern containing `*` acts as a **prefix wildcard** (`"steel*"` matches any reference product starting with `steel`); a pattern without `*` must be an **exact match**. `excludes` follows the same wildcard rules and overrides any `matchings` hit.

Then pass your file to the setup:

```python
setup_lca_supply_mat(
    target_db="ecoinvent-3.11-cutoff",
    yaml_file="path/to/my_definitions.yaml")
```
